# Building RAG Chatbots with LangChain

In this example, we'll work on building an AI chatbot from start-to-finish. We will be using LangChain, OpenAI, and Pinecone vector DB, to build a chatbot capable of learning from the external world using **R**etrieval **A**ugmented **G**eneration (RAG).

We will be using a dataset sourced from ArXiv papers to help our chatbot answer questions about the latest and greatest in the world of GenAI.

By the end of the example we'll have a functioning chatbot and RAG pipeline that ***can hold a conversation and provide informative responses based on a knowledge base.***

### Before you begin

You'll need to get an [API key](https://platform.openai.com/account/api-keys) of your AI model and an [Pinecone API key](https://app.pinecone.io).

### Prerequisites

Before we start building our chatbot, we need to install some Python libraries. Here's a brief overview of what each library does:

- **langchain**: This is a library for GenAI. We'll use it to chain together different language models and components for our chatbot.
- **datasets**: This library provides a vast array of datasets for machine learning. We'll use it to load our knowledge base for the chatbot.
- **pinecone**: This is the official Pinecone Python client. We'll use it to interact with the Pinecone API and store our chatbot's knowledge base in a vector database.

You can install these libraries using pip like so:

In [1]:
pip install -qU pinecone langchain-community langchain-openai langchain-google-genai langchain-pinecone langchain_huggingface tiktoken

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os

In [3]:
import time

In [4]:
from tqdm.auto import tqdm  # for progress bar

In [5]:
from datasets import load_dataset

In [6]:
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
os.environ['GOOGLE_GENAI_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [7]:
from langchain_core.chat_history import (
    BaseChatMessageHistory, # Interface/typing for pluggable chat history backends.
    InMemoryChatMessageHistory # for in-memory chat history for the session.
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # for build chat-style prompts; placeholder to inject prior messages/history.
from langchain_core.runnables.history import RunnableWithMessageHistory # to wrap a chain so it automatically reads/writes chat history per session id.
from langchain_text_splitters import RecursiveCharacterTextSplitter # doc splits into overlapping chunks to improve retrieval quality.
from langchain_community.document_loaders import PyPDFLoader # for loading and parsing pdf files in to lanchain doc objects

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_pinecone import PineconeVectorStore

/tmp/ipykernel_921/834599027.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader # for loading and parsing pdf files in to lanchain doc objects


In [8]:
from pinecone import Pinecone, ServerlessSpec

### Building a Chatbot (no RAG)

We will be relying heavily on the LangChain library to bring together the different components needed for our chatbot. To begin, we'll create a simple chatbot without any retrieval augmentation.

In [9]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [10]:
chat = ChatGoogleGenerativeAI(
    api_key=os.environ["GOOGLE_GENAI_API_KEY"],
    model="gemini-3.1-flash-lite"
)

# chat = ChatOpenAI(
#     api_key=os.environ["OPENAI_API_KEY"],
#     model='gpt-5.4-mini-2026-03-17'
# )

Chats with AI chat models are typically structured (in plain text) like this:

```
System: You are a helpful assistant.

User: Hi AI, how are you today?

Assistant: I'm great thank you. How can I help you?

User: I'd like to understand string theory.

Assistant:
```

The final `"Assistant:"` without a response is what would prompt the model to continue the conversation. In the official OpenAI `ChatCompletion` endpoint these would be passed to the model in a format like:

```python
[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi AI, how are you today?"},
    {"role": "assistant", "content": "I'm great thank you. How can I help you?"}
    {"role": "user", "content": "I'd like to understand string theory."}
]
```

In LangChain there is a slightly different format. We use three _message_ objects like so:

In [11]:
messages = [
    SystemMessage(content="You are a helpful assistant. Please make your responses brief."),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

The format is very similar, we just swapped the role of `"user"` for `HumanMessage`, and the role of `"assistant"` for `AIMessage`.

We generate the next response from the AI by passing these messages to the `ChatOpenAI` object.

In [12]:
res = chat.invoke(messages)
res

AIMessage(content=[{'type': 'text', 'text': 'String theory suggests that the fundamental building blocks of the universe are not point-like particles, but tiny, vibrating "strings."\n\nDifferent vibrational frequencies of these strings manifest as different particles (like electrons or quarks). The theory aims to be a "Theory of Everything" by reconciling quantum mechanics with general relativity, though it requires the existence of extra spatial dimensions beyond the three we experience.', 'extras': {'signature': 'EnMKcQFpFH0T1VMazZ6nW+jufADNj3MYWI50lLfinCiWEiqYtpUF0t3+gvZzqjWx9BtfFnFiFid5s0ZWs5mubZKarGLkANL/ZM6E6nemd84RGldxprBrmA6jQYg0cQ9raE13U5hsow5OFLizEqCUJcbepFmX'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1a85d-ad26-77cc-bf9e-418ff94dcd51-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 76, 'total_

In response we get another AI message object. We can print it more clearly like so:

In [13]:
print(res.content[0]['text'])

String theory suggests that the fundamental building blocks of the universe are not point-like particles, but tiny, vibrating "strings."

Different vibrational frequencies of these strings manifest as different particles (like electrons or quarks). The theory aims to be a "Theory of Everything" by reconciling quantum mechanics with general relativity, though it requires the existence of extra spatial dimensions beyond the three we experience.


Because `res` is just another `AIMessage` object, we can append it to `messages`, add another `HumanMessage`, and generate the next response in the conversation.

In [14]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)

# add to messages
messages.append(prompt)

# send to AI model
res = chat.invoke(messages)

In [15]:
print(res.content[0]['text'])

Physicists believe it can be a "Theory of Everything" because it is one of the few mathematical frameworks that successfully incorporates **gravity** (General Relativity) into the same quantum mechanical language used for the other fundamental forces (electromagnetism, weak, and strong nuclear forces). 

By replacing zero-dimensional particles with strings, the theory naturally includes the "graviton"—the particle responsible for gravity—which previous quantum models could not do without resulting in mathematical "infinities" or errors.


### Dealing with Hallucinations

We have our chatbot, but as mentioned — the knowledge of LLMs can be limited. The reason for this is that LLMs learn all they know during training. An LLM essentially compresses the "world" as seen in the training data into the internal parameters of the model. We call this knowledge the _parametric knowledge_ of the model.

By default, LLMs have no access to the external world.

The result of this is very clear when we ask LLMs about more recent information, like about the Llama 2 LLM.

In [16]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="What is so special about Gemma 4?"
)

# add to messages
messages.append(prompt)

# send to model
res = chat.invoke(messages)

In [17]:
print(res.content[0]['text'])

It appears there may be a slight misunderstanding: **there is no "Gemma 4"** released or announced by Google at this time.

The current generation of Google's open-weights models is **Gemma 2** (available in sizes like 2B, 9B, and 27B). These models are notable for their efficiency, high performance relative to their size, and use of "distillation" techniques to pack the knowledge of larger models into smaller, more accessible ones.


The Gemini 3.1 Flash Lite model has a knowledge cutoff date of January 2025. This means it doesn't have information on developments that occurred after that time, including the release of Google's Gemma 4 in April 2026. Therefore, Gemini 3.1 Flash Lite would not be aware of Gemma 4 or any related advancements.



Our chatbot can no longer help us, it doesn't contain the information we need to answer the question. It was very clear from this answer that the LLM doesn't know the informaiton, but sometimes an LLM may respond like it _does_ know the answer — and this can be very hard to detect.

In [18]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Can you tell me about Matryoshka Formers?"
)
# add to messages
messages.append(prompt)

# send to AI
res = chat.invoke(messages)

In [19]:
print(res.content[0]['text'])

"Matryoshka Formers" refers to a technique in machine learning inspired by Russian nesting dolls. It involves training models—specifically Large Language Models—to produce **nested representations**.

The core idea is that a model can represent information at different levels of granularity within the same vector. This allows the model to be truncated or compressed for different tasks without needing to be retrained:

*   **Efficiency:** You can use a smaller portion of the vector for simple, fast tasks (like retrieval) and the full vector for complex, high-accuracy reasoning.
*   **Flexibility:** It enables a single model to adapt its computational cost dynamically based on the complexity of the query.

Essentially, it allows a single model to provide high-quality outputs at varying "depths" of computation.


There is another way of feeding knowledge into LLMs. It is called _source knowledge_ and it refers to any information fed into the LLM via the prompt. We can try that with the LLMChain question. We can take a description of this object from the LangChain documentation.

In [20]:
matformer_information = [
    "MatFormer (short for Matryoshka Transformer) is a natively elastic model architecture that incorporates nested parameter sub-structures directly inside standard transformer blocks.",
    "By optimizing these nested structures—primarily within the Feed-Forward Network layers—a single base model can dynamically yield hundreds of accurate smaller sub-models without requiring separate training runs.",
    "This elastic design allows deployment systems to slice and adjust parameter sizes on the fly to meet real-time compute, memory, and latency constraints."
]

source_knowledge = "\n".join(matformer_information)

We can feed this additional knowledge into our prompt with some instructions telling the LLM how we'd like it to use this information alongside our original query.

In [21]:
query = "Can you tell me about Matryoshka Formers?"

augmented_prompt = f"""Using the contexts below, answer the query.

Contexts:
{source_knowledge}

Query: {query}"""

Now we feed this into our chatbot as we were before.

In [22]:
# create a new user prompt
prompt = HumanMessage(
    content=augmented_prompt
)
# add to messages
messages.append(prompt)

# send to AI
res = chat.invoke(messages)

In [23]:
print(res.content[0]['text'])

Matryoshka Formers (MatFormer) are an elastic model architecture featuring nested parameter sub-structures within standard transformer blocks. By optimizing these nested structures—mainly in the Feed-Forward Network layers—a single base model can dynamically generate hundreds of smaller, accurate sub-models. This allows systems to adjust size and compute requirements in real-time without needing separate training runs.


The quality of this answer is phenomenal. This is made possible thanks to the idea of augmented our query with external knowledge (source knowledge). There's just one problem — how do we get this information in the first place?

We will use Pinecone and vector databases. But first, we'll need a dataset.

### Importing the Data

In this task, we will be importing our data. We will be using the Hugging Face Datasets library to load our data. Specifically, we will be using the `yufan/arxiv-metadata-2020-2026` dataset. This dataset contains a collection of ArXiv papers which will serve as the external knowledge base for our chatbot.

In [24]:
!mkdir data

In [25]:
dataset = load_dataset(
    "yufan/arxiv-metadata-2020-2026",
    "Computer_Science",
    split="2023",
)

README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

Computer_Science/2020/metadata.jsonl: reconstructing file:   0%|          |  0.00B /  116MB            

Computer_Science/2020/metadata.jsonl: downloading bytes:           |  0.00B            

Computer_Science/2021/metadata.jsonl: reconstructing file:   0%|          |  0.00B /  126MB            

Computer_Science/2021/metadata.jsonl: downloading bytes:           |  0.00B            

Computer_Science/2022/metadata.jsonl: reconstructing file:   0%|          |  0.00B /  135MB            

... (12 outputs left out)

Generating 2023 split: 0 examples [00:00, ? examples/s]

Generating 2024 split: 0 examples [00:00, ? examples/s]

Generating 2025 split: 0 examples [00:00, ? examples/s]

Generating 2026 split: 0 examples [00:00, ? examples/s]

In [26]:
dataset

Dataset({
    features: ['arxiv_id', 'corpusId', 'title', 'authors', 'year', 'publicationDate', 'citationCount', 'influentialCitationCount', 'fieldsOfStudy', 'abstract', 'fetched_at', 'citation_refreshed_at', 'citation_source_release', 'company'],
    num_rows: 93422
})

In [27]:
dataset[0]

{'arxiv_id': '2301.00001',
 'corpusId': 255373121,
 'title': 'NFTrig',
 'authors': 'Jordan J. Thompson; Ryan Benac; Kidus Olana; Talha Hassan; Andrew Sward; Tauheed Khan Mohd',
 'year': 2022,
 'publicationDate': datetime.datetime(2022, 12, 21, 0, 0),
 'citationCount': 0,
 'influentialCitationCount': 0,
 'fieldsOfStudy': 'Computer Science',
 'abstract': 'NFTrig is a web-based application created for use as an educational tool to teach trigonometry and block chain technology. Creation of the application includes front and back end development as well as integration with other outside sources including MetaMask and OpenSea. The primary development languages include HTML, CSS (Bootstrap 5), and JavaScript as well as Solidity for smart contract creation. The application itself is hosted on Moralis utilizing their Web3 API. This technical report describes how the application was created, what the application requires, and smart contract design with security considerations in mind. The NFTrig

#### Dataset Overview

The dataset we are using is sourced from the Llama 2 ArXiv papers. It is a collection of academic papers from ArXiv, a repository of electronic preprints approved for publication after moderation. Each entry in the dataset represents a "chunk" of text from these papers.

Because most **L**arge **L**anguage **M**odels (LLMs) only contain knowledge of the world as it was during training, they cannot answer our questions about Llama 2 — at least not without this data.

### Task 4: Building the Knowledge Base

We now have a dataset that can serve as our chatbot knowledge base. Our next task is to transform that dataset into the knowledge base that our chatbot can use. To do this we must use an embedding model and vector database.

We begin by initializing our connection to Pinecone, this requires a [free API key](https://app.pinecone.io).

In [28]:
# configure client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [29]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

In [30]:
embed_model = "mixedbread-ai/mxbai-embed-large-v1"
embedding = HuggingFaceEmbeddings(
    model_name=embed_model,
    model_kwargs={'device': 'cuda:0'}
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  670MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Using this model we can create embeddings like so:

In [31]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embedding.embed_documents(texts)
len(res), len(res[0])

(2, 1024)

In [32]:
dimensions = len(res[0])

Then we initialize the index. We will be using the `all-MiniLM-L6-v2` model for creating the embeddings, so we set the `dimension` to `384`.

In [33]:
index_name = 'arxiv-rag' # data structure that organizes high-dimensional vectors to enable very fast retrievval/ similarity searches
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=dimensions,  # dimensionality of all-MiniLM
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

From this we get two (aligning to our two chunks of text) 1536-dimensional embeddings.

We're now ready to embed and index all our our data! We do this by looping through our dataset and embedding and inserting everything in batches.

In [34]:
dataset[0]

{'arxiv_id': '2301.00001',
 'corpusId': 255373121,
 'title': 'NFTrig',
 'authors': 'Jordan J. Thompson; Ryan Benac; Kidus Olana; Talha Hassan; Andrew Sward; Tauheed Khan Mohd',
 'year': 2022,
 'publicationDate': datetime.datetime(2022, 12, 21, 0, 0),
 'citationCount': 0,
 'influentialCitationCount': 0,
 'fieldsOfStudy': 'Computer Science',
 'abstract': 'NFTrig is a web-based application created for use as an educational tool to teach trigonometry and block chain technology. Creation of the application includes front and back end development as well as integration with other outside sources including MetaMask and OpenSea. The primary development languages include HTML, CSS (Bootstrap 5), and JavaScript as well as Solidity for smart contract creation. The application itself is hosted on Moralis utilizing their Web3 API. This technical report describes how the application was created, what the application requires, and smart contract design with security considerations in mind. The NFTrig

In [36]:
data = dataset.to_pandas()  # this makes it easier to iterate over the dataset
data = data[~(data['abstract'].isna() | data['title'].isna())]

batch_size = 200

for i in tqdm(range(0, len(data), batch_size)):
    i_end = min(len(data), i+batch_size)
    # get batch of data
    batch = data.iloc[i:i_end]
    # generate unique ids for each chunk
    ids = [f"{x['corpusId']}-{x['arxiv_id']}" for i, x in batch.iterrows()]
    # get text to embed
    texts = [x['title'] + '\n\n' + x['abstract'] for _, x in batch.iterrows()]
    # embed text
    embeds = embedding.embed_documents(texts)
    # get metadata to store in Pinecone
    metadata = [
        {'abstract': x['abstract'],
         'arxiv_id': x['arxiv_id'],
         'title': x['title']} for i, x in batch.iterrows()
    ]
    # add to Pinecone
    index.upsert(vectors=zip(ids, embeds, metadata))

  0%|          | 0/468 [00:00<?, ?it/s]

We can check that the vector index has been populated using `describe_index_stats` like before:

In [37]:
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 93421}},
 'total_vector_count': 93421,
 'vector_type': 'dense'}

#### Retrieval Augmented Generation

We've built a fully-fledged knowledge base. Now it's time to connect that knowledge base to our chatbot. To do that we'll be diving back into LangChain and reusing our template prompt from earlier.

To use LangChain with your data, we need to tell LangChain where your data is stored and how to access it. This is done through a "vectorstore" abstraction.
We pass in our vector `index` to initialize the object.

In [38]:
# initialize the vector store object
vectorstore = PineconeVectorStore(
    index, embedding, "abstract"
)

Using this `vectorstore` we can already query the index and see if we have any relevant information given our question about Llama 2.

In [39]:
query = "What is a MatFormer?"

vectorstore.similarity_search(query, k=3)

[Document(id='257378508-2303.03800', metadata={'arxiv_id': '2303.03800', 'title': 'Lformer: Text-to-Image Generation with L-shape Block Parallel Decoding'}, page_content='Generative transformers have shown their superiority in synthesizing high-fidelity and high-resolution images, such as good diversity and training stability. However, they suffer from the problem of slow generation since they need to generate a long token sequence autoregressively. To better accelerate the generative transformers while keeping good generation quality, we propose Lformer, a semi-autoregressive text-to-image generation model. Lformer firstly encodes an image into $h{\\times}h$ discrete tokens, then divides these tokens into $h$ mirrored L-shape blocks from the top left to the bottom right and decodes the tokens in a block parallelly in each step. Lformer predicts the area adjacent to the previous context like autoregressive models thus it is more stable while accelerating. By leveraging the 2D structure

We return a lot of text here and it's not that clear what we need or what is relevant. Fortunately, our LLM will be able to parse this information much faster than us. All we need is to connect the output from our `vectorstore` to our `chat` chatbot. To do that we can use the same logic as we used earlier.

In [40]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = vectorstore.similarity_search(query, k=3)
    # get the text from the results
    source_knowledge = "\n".join([x.page_content for x in results])
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

Using this we produce an augmented prompt:

In [41]:
print(augment_prompt(query))

Using the contexts below, answer the query.

    Contexts:
    Generative transformers have shown their superiority in synthesizing high-fidelity and high-resolution images, such as good diversity and training stability. However, they suffer from the problem of slow generation since they need to generate a long token sequence autoregressively. To better accelerate the generative transformers while keeping good generation quality, we propose Lformer, a semi-autoregressive text-to-image generation model. Lformer firstly encodes an image into $h{\times}h$ discrete tokens, then divides these tokens into $h$ mirrored L-shape blocks from the top left to the bottom right and decodes the tokens in a block parallelly in each step. Lformer predicts the area adjacent to the previous context like autoregressive models thus it is more stable while accelerating. By leveraging the 2D structure of image tokens, Lformer achieves faster speed than the existing transformer-based methods while keeping goo

There is still a lot of text here, so let's pass it onto our chat model to see how it performs.

In [42]:
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages.append(prompt)

res = chat.invoke(messages)

In [43]:
print(res.content[0]['text'])

A **MatFormer** (Matryoshka Transformer) is an elastic model architecture designed to address the challenge of deploying models across diverse hardware constraints. 

It works by incorporating nested parameter structures—specifically within the Feed-Forward Network (FFN) layers—directly into a standard transformer. This allows a single base model to contain hundreds of smaller, accurate sub-models that can be "sliced" and deployed on the fly to meet specific latency, memory, or cost requirements without the need for additional training runs.


We can continue with more questions. Let's try _without_ RAG first:

In [46]:
prompt = HumanMessage(
    content="What is SESNet?"
)

res = chat.invoke(messages + [prompt])

In [47]:
print(res.content[0]['text'])

Based on the provided text:

*   **MatFormer** (Matryoshka Transformer) is an elastic model architecture that incorporates nested parameter structures (specifically within Feed-Forward Network layers) into a standard Transformer. This allows a single base model to dynamically provide hundreds of accurate smaller sub-models, enabling flexible deployment across different latency, cost, and compute constraints without needing separate training runs.

*   **SESNet** is not mentioned in the provided contexts.


Let's try again but with RAG.

In [49]:
prompt = HumanMessage(
    content=augment_prompt(
        "What is SESNet?"
    )
)

res = chat.invoke(messages + [prompt])

In [50]:
print(res.content[0]['text'])

Based on the provided contexts:

*   **MatFormer (Matryoshka Transformer):** It is a natively elastic Transformer architecture designed for diverse deployment constraints. By incorporating nested Feed-Forward Network (FFN) blocks within a standard Transformer, a single base model can dynamically yield hundreds of accurate smaller sub-models. This allows for optimized inference across different latency and memory requirements without needing to train separate models.

*   **SESNet:** It is a supervised deep-learning model designed to predict the functional fitness of protein mutants. It integrates sequence and structure information by utilizing local evolutionary context, global evolutionary context, and microenvironment residue data. It also employs a data augmentation strategy using unsupervised pre-training, allowing it to achieve high accuracy on higher-order protein mutants with very little experimental data.


Delete the index to save resources:

In [51]:
pc.delete_index(index_name)

---